In [1]:

import random
from collections import deque
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

env = gym.make("LunarLander-v3") # Changed from LunarLander-v2 to LunarLander-v3

print("State Space Shape:", env.observation_space.shape[0])  # Continuous (8 variables)
print("Number of actions:", env.action_space.n)  # Discrete (4 actions)


GAMMA = 0.99
LR = 1e-3
BATCH_SIZE = 64
MEMORY_SIZE = 50000
EPSILON_DECAY = 0.995
MIN_EPSILON = 0.01


class QNetwork(nn.Module):
    """Neural Network that evaluates the value of each action given a state."""

    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
        )

    def forward(self, x):
        return self.fc(x)

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

policy_net = QNetwork(state_dim, action_dim)
target_net = QNetwork(state_dim, action_dim)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=LR)
memory = deque(maxlen=MEMORY_SIZE)
epsilon = 1.0

def select_action(state, epsilon):
    """Epsilon-greedy policy step to balance exploration and exploitation."""
    if random.random() < epsilon:
        return env.action_space.sample()  # Explore

    state_t = torch.FloatTensor(state).unsqueeze(0)
    with torch.no_grad():
        q_values = policy_net(state_t)
    return torch.argmax(q_values).item()  # Exploit (best action)


def optimize_model():
    """Performs an optimization step on the neural network values (Bellman Update)."""
    if len(memory) < BATCH_SIZE:
        return

    batch = random.sample(memory, BATCH_SIZE)
    states, actions, rewards, next_states, dones = zip(*batch)

    states_t = torch.FloatTensor(np.array(states))
    actions_t = torch.LongTensor(actions).unsqueeze(1)
    rewards_t = torch.FloatTensor(rewards).unsqueeze(1)
    next_states_t = torch.FloatTensor(np.array(next_states))
    dones_t = torch.FloatTensor(dones).unsqueeze(1)

    current_q = policy_net(states_t).gather(1, actions_t)


    with torch.no_grad():
        max_next_q = target_net(next_states_t).max(1)[0].unsqueeze(1)
        target_q = rewards_t + (GAMMA * max_next_q * (1 - dones_t))


    loss = nn.MSELoss()(current_q, target_q)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


print("\n--- Starting Training Loop ---")
episodes = 300

for episode in range(episodes):
    state, info = env.reset()
    total_reward = 0

    while True:
        action = select_action(state, epsilon)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated


        memory.append((state, action, reward, next_state, done))

        state = next_state
        total_reward += reward


        optimize_model()

        if done:
            break


    epsilon = max(MIN_EPSILON, epsilon * EPSILON_DECAY)


    if episode % 10 == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if (episode + 1) % 50 == 0:
        print(f"Episode {episode+1}/{episodes} | Total Reward: {total_reward:.2f} | Epsilon: {epsilon:.2f}")

env.close()


print("\n--- Rendering Trained Agent ---")
eval_env = gym.make("LunarLander-v3", render_mode="human") # Changed from LunarLander-v2 to LunarLander-v3
state, info = eval_env.reset()

for _ in range(500):
    # Set epsilon to 0 to strictly use optimal network predictions
    action = select_action(state, epsilon=0.0)
    state, reward, terminated, truncated, info = eval_env.step(action)

    if terminated or truncated:
        state, info = eval_env.reset()

eval_env.close()


State Space Shape: 8
Number of actions: 4

--- Starting Training Loop ---
Episode 50/300 | Total Reward: -118.28 | Epsilon: 0.78
Episode 100/300 | Total Reward: -56.93 | Epsilon: 0.61
Episode 150/300 | Total Reward: 29.18 | Epsilon: 0.47
Episode 200/300 | Total Reward: -48.48 | Epsilon: 0.37
Episode 250/300 | Total Reward: -16.24 | Epsilon: 0.29
Episode 300/300 | Total Reward: 204.26 | Epsilon: 0.22

--- Rendering Trained Agent ---


In [ ]:
# Install necessary dependencies for Box2D environments
!pip install swig
!pip install "gymnasium[box2d]"

# Restart the runtime after installing new packages
import os
os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 23.2 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 31.1 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
